# Demo 4 (new way) — Training from a Unified Studio project

The **same** custom `train.py` and the **same** `SKLearn` estimator as the old
way — but launched from a governed project. Note what you *don't* do here: no
role ARN to paste (the project provides it), and the run is tracked with lineage
back to the input dataset.

_Cross-reference: same script as [old way](../WALKTHROUGH.md); features from
[PIPELINE_SPEC](../../PIPELINE_SPEC.md) T16._

In [ ]:
import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker.sklearn.estimator import SKLearn

# In Unified Studio these come from the project context — no ARNs to paste.
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = "roi-smdemo-029331796573-us-east-2"
prefix = "ml/taxi"

In [ ]:
estimator = SKLearn(
    entry_point="train.py",
    source_dir="..",            # the folder containing train.py
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    sagemaker_session=sess,
    base_job_name="roi-smdemo-taxi-train-us",
    hyperparameters={"learning-rate": 0.1, "n-estimators": 150, "max-depth": 3},
    metric_definitions=[
        {"Name": "validation:accuracy", "Regex": "validation_accuracy=([0-9\\.]+)"},
        {"Name": "validation:auc", "Regex": "validation_auc=([0-9\\.]+)"},
    ],
)

In [ ]:
estimator.fit({
    "train": TrainingInput(f"s3://{bucket}/{prefix}/train/", content_type="text/csv"),
    "validation": TrainingInput(f"s3://{bucket}/{prefix}/validation/", content_type="text/csv"),
}, logs=True)
print("model artifact:", estimator.model_data)

Open the project's **Compute → Training jobs** (or the experiment view) to see
this run tracked with lineage — the governance you'd otherwise wire up by hand.